## TUGAS MANDIRI (Dikerjakan Selama 1 Minggu)

> **Tenggat waktu:** dikumpulkan paling lambat **sebelum Pertemuan 6 dimulai**.
> **Sifat tugas:** individu.

### Konteks / Skenario

Manajemen platform e-commerce meminta dibuatkan **dashboard performa cabang toko** yang menggabungkan data transaksi (yang sudah ada di HDFS sejak Pertemuan 3-4) dengan data referensi target penjualan tiap cabang. Anda ditugaskan menyiapkan analisis ini menggunakan kombinasi **join, window function, dan Spark SQL** — persis seperti yang dipelajari hari ini.

### Menyiapkan Dataset

Jalankan cell berikut untuk membuat **dua tabel** dan mengunggah tabel transaksi ke HDFS (tabel target cukup dibuat langsung sebagai Spark DataFrame, karena berukuran kecil dan jarang berubah — praktik umum untuk tabel referensi/*dimension table*).

In [11]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, row_number
from pyspark.sql.window import Window
import pandas as pd

# Inisialisasi SparkSession
spark = SparkSession.builder \
    .appName("Tugas5_BigData") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("SparkSession berhasil dibuat!")

26/09/23 23:51:33 WARN Utils: Your hostname, aramsamsam-VirtualBox resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/09/23 23:51:33 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/23 23:51:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/23 23:51:39 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


SparkSession berhasil dibuat!


In [1]:
# Sel ini membuat dataset baru untuk Tugas Mandiri Pertemuan 4 dan mengunggahnya ke HDFS
import numpy as np
import pandas as pd

np.random.seed(99)
n = 1000
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga", "Olahraga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo", "Kebumen"]
metode_bayar_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]
tanggal_range = pd.date_range("2026-09-01", "2026-09-30", freq="D")

data = {
    "order_id": [f"ORD-{3000 + i}" for i in range(n)],
    "tanggal": np.random.choice(tanggal_range, size=n).astype(str),
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 12, size=n),
    "harga_satuan": np.random.choice([20000, 45000, 60000, 90000, 125000, 200000, 350000], size=n),
    "metode_pembayaran": np.random.choice(metode_bayar_list, size=n),
    "rating": np.random.choice([1, 2, 3, 4, 5, np.nan], size=n, p=[0.03, 0.02, 0.10, 0.30, 0.35, 0.20]),
}
df_tugas4 = pd.DataFrame(data)
df_tugas4.to_csv("transaksi_september_2026.csv", index=False)
print(f"Dataset dibuat: {df_tugas4.shape[0]} baris")

# Mengunggah ke HDFS
!hdfs dfs -mkdir -p /user/mahasiswa/tugas4
!hdfs dfs -put -f transaksi_september_2026.csv /user/mahasiswa/tugas4/
print("Berhasil diunggah ke HDFS: /user/mahasiswa/tugas4/transaksi_september_2026.csv")

Dataset dibuat: 1000 baris
Berhasil diunggah ke HDFS: /user/mahasiswa/tugas4/transaksi_september_2026.csv


In [7]:
import numpy as np
import pandas as pd

# 1. Tabel target cabang (referensi)
data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"],
}

# 2. Generasi data transaksi
np.random.seed(55)
n = 500
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"]

data_transaksi_t5 = {
    "order_id": [f"TRX-{i}" for i in range(n)],
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 10, size=n),
    "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000], size=n),
}

# Simpan ke file CSV di folder lokal
pd.DataFrame(data_transaksi_t5).to_csv("transaksi_tugas5.csv", index=False)
print("File transaksi_tugas5.csv berhasil dibuat di folder lokal!")

File transaksi_tugas5.csv berhasil dibuat di folder lokal!


In [13]:
!hdfs dfs -mkdir -p /user/mahasiswa/tugas5
!hdfs dfs -put -f transaksi_tugas5.csv /user/mahasiswa/tugas5/

In [14]:
# Membaca data transaksi dari HDFS
# Membaca data transaksi dari HDFS menggunakan URL HDFS lengkap
df_transaksi = spark.read.csv(
    "hdfs://localhost:9000/user/mahasiswa/tugas5/transaksi_tugas5.csv", 
    header=True, 
    inferSchema=True
)

df_transaksi = df_transaksi.withColumn("pendapatan", col("unit_terjual") * col("harga_satuan"))

# Tabel Target Cabang
data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"],
}
df_target = spark.createDataFrame(pd.DataFrame(data_target_cabang))

### Instruksi Pengerjaan

Buat notebook baru **`Tugas5_[NPM]_[Nama Lengkap].ipynb`**, buat `SparkSession`, lalu:
1. Baca `transaksi_tugas5.csv` **dari HDFS** menjadi `df_transaksi`, tambahkan kolom `pendapatan` (`unit_terjual x harga_satuan`).
2. Buat `df_target` dari dictionary `data_target_cabang` di atas

Kerjakan bagian **A sampai D** berikut:

---

**A. Join & Perbandingan Target** *(bobot 25%)*

Ringkas total `pendapatan` per `kota` dari `df_transaksi`, lalu **join** dengan `df_target`. Tambahkan kolom `pencapaian_persen`. Urutkan hasil dari pencapaian tertinggi.



In [15]:
# Perintah bash untuk membuat direktori & upload file ke HDFS
ringkasan_kota = df_transaksi.groupBy("kota").agg(spark_sum("pendapatan").alias("total_pendapatan"))

hasil_a = ringkasan_kota.join(df_target, on="kota", how="inner") \
    .withColumn("pencapaian_persen", (col("total_pendapatan") / col("target_bulanan") * 100)) \
    .orderBy(col("pencapaian_persen").desc())

hasil_a.show()

+----------+----------------+--------------+----------+------------------+
|      kota|total_pendapatan|target_bulanan|pic_cabang| pencapaian_persen|
+----------+----------------+--------------+----------+------------------+
| Purworejo|        45650000|      30000000|     Fitri|152.16666666666669|
|      Solo|        33475000|      40000000|      Bayu|           83.6875|
|Yogyakarta|        47275000|      60000000|      Joko| 78.79166666666667|
|  Magelang|        31650000|      45000000|      Rani| 70.33333333333334|
|  Semarang|        38175000|      55000000|      Sari|  69.4090909090909|
+----------+----------------+--------------+----------+------------------+



**B. Window Function — Kategori Terlaris per Kota** *(bobot 25%)*

Menggunakan window function, tentukan **kategori dengan pendapatan tertinggi di setiap kota** (top-1 saja, gunakan `row_number()`).



In [16]:
df_kat_kota = df_transaksi.groupBy("kota", "kategori").agg(spark_sum("pendapatan").alias("total_pendapatan"))
win_kat = Window.partitionBy("kota").orderBy(col("total_pendapatan").desc())

hasil_b = df_kat_kota.withColumn("rn", row_number().over(win_kat)).filter(col("rn") == 1)
hasil_b.show()

[Stage 6:>                                                          (0 + 1) / 1]

+----------+--------------------+----------------+---+
|      kota|            kategori|total_pendapatan| rn|
+----------+--------------------+----------------+---+
|  Magelang|Kesehatan & Kecan...|         7275000|  1|
| Purworejo|Kesehatan & Kecan...|        10075000|  1|
|  Semarang|        Rumah Tangga|        11125000|  1|
|      Solo|Kesehatan & Kecan...|         8425000|  1|
|Yogyakarta|             Fashion|        13325000|  1|
+----------+--------------------+----------------+---+



**C. Spark SQL** *(bobot 25%)*

Daftarkan `df_transaksi` dan `df_target` sebagai *temporary view*, lalu **tulis satu kueri SQL** (bukan DataFrame API) yang menampilkan: `kota`, `pic_cabang`, dan jumlah transaksi (`COUNT`) di kota tersebut, diurutkan dari jumlah transaksi terbanyak.




In [18]:
df_transaksi.createOrReplaceTempView("transaksi_t5")
df_target.createOrReplaceTempView("target_t5")

hasil_c = spark.sql("""
    SELECT t.kota, tg.pic_cabang, COUNT(t.order_id) AS jumlah_transaksi
    FROM transaksi_t5 t
    JOIN target_t5 tg ON t.kota = tg.kota
    GROUP BY t.kota, tg.pic_cabang
    ORDER BY jumlah_transaksi DESC
""")
hasil_c.show()

[Stage 19:>                                                         (0 + 4) / 4]

+----------+----------+----------------+
|      kota|pic_cabang|jumlah_transaksi|
+----------+----------+----------------+
| Purworejo|     Fitri|             116|
|Yogyakarta|      Joko|             110|
|      Solo|      Bayu|              95|
|  Semarang|      Sari|              93|
|  Magelang|      Rani|              86|
+----------+----------+----------------+



**D. Kesimpulan** *(bobot 25%)*

Tulis pada markdown cell (**minimal 100 kata**): berdasarkan hasil bagian A dan B, **cabang mana yang berkinerja paling baik** dan **cabang mana yang paling perlu perhatian manajemen**? Sertakan angka-angka pendukung dari hasil analisis kalian, bukan opini tanpa dasar data.




Berdasarkan hasil agregasi data transaksi e-commerce dan pencapaian target cabang:

* Cabang Berkinerja Paling Baik:

    Cabang [Nama Kota, contoh: Purworejo/Magelang] mencatatkan kinerja terbaik dengan tingkat pencapaian target sebesar [Contoh: 112.5%] dari target bulanan sebesar [Contoh: Rp30.000.000]. Keberhasilan ini didorong oleh tingginya kontribusi penjualan pada kategori [Nama Kategori Terlaris dari Bagian B] yang menyumbang pendapatan sebesar [Contoh: Rp12.500.000] dari total [Contoh: 105] transaksi yang ditangani oleh PIC [Nama PIC].

* Cabang yang Membutuhkan Perhatian Manajemen:

    Cabang [Nama Kota, contoh: Yogyakarta/Semarang] membutuhkan perhatian khusus dan evaluasi strategi dari manajemen. Cabang ini hanya mencapai [Contoh: 65.4%] dari target bulanan sebesar [Contoh: Rp60.000.000]. Meskipun volume transaksinya tergolong sedang, rata-rata nilai transaksi per pelanggan masih rendah. Kategori utama seperti [Kategori] belum mencapai penetrasi pasar yang optimal.

---
### Ketentuan Pengumpulan

- Kumpulkan `Tugas5_[NPM]_[Nama Lengkap].ipynb` melalui Asisten Praktikum, paling lambat **1 minggu dari hari ini, pukul 23.59 WIB**.
- Pastikan Hadoop aktif dan seluruh cell sudah dijalankan (**Run All**) sebelum dikumpulkan.
- Bagian A & B **wajib** menggunakan DataFrame API; bagian C **wajib** menggunakan Spark SQL murni (`spark.sql(...)`).

### Rubrik Penilaian

| Bagian | Kriteria | Bobot |
|---|---|---|
| A. Join & Pencapaian Target | Join benar, kolom `pencapaian_persen` terhitung tepat | 25% |
| B. Window Function | Kategori terlaris per kota teridentifikasi dengan benar menggunakan `row_number()` | 25% |
| C. Spark SQL | Kueri SQL berjalan benar & menghasilkan output yang sesuai instruksi | 25% |
| D. Kesimpulan | Analisis berbasis data, jelas menyebutkan cabang terbaik & yang perlu perhatian | 25% |